In [2]:
# -*- coding: utf-8 -*-
"""
Step 4 OPTIMIZED: Internal Validation (10-fold CV)
Focus: Evaluating the Optimized Stacking Ensemble rigorously on Training Data
Output: Figure 3 (JAMA Style) & Validation Table
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast 
import warnings
from scipy import stats
from docx import Document
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, roc_curve, auc
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.base import BaseEstimator, ClassifierMixin, clone

# --- Plotting Style ---
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 300
warnings.filterwarnings('ignore')

# --- Configuration ---
BASE_DIR = os.getcwd()
TRAIN_FILE = os.path.join(BASE_DIR, 'imputation', 'imputed_data', 'train_imputed_random_forest.csv')
PARAMS_FILE = os.path.join(BASE_DIR, 'Tuning_TwoStage_Results', 'best_hyperparameters_twostage.xlsx')
RFE_LISTS_FOLDER = os.path.join(BASE_DIR, 'RFE_Final_Run', 'feature_lists')
OUTPUT_DIR = os.path.join(BASE_DIR, 'Final_Internal_Validation_Optimized')

TARGET_COL = 'PostopAKI'
SEED = 42

COLORS = {
    'LR': '#D55E00', 'DT': '#0072B2', 'RF': '#009E73', 
    'SVM': '#CC79A7', 'KNN': '#F0E442', 'NB': '#56B4E9', 
    'XGB': '#E69F00', 'SGBT': '#999999', 'NNET': '#000000',
    'Voting': '#882255', 'Stacking': '#117733'
}

# --- Helper Classes ---
class NNETWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_layer_sizes=(100,), alpha=0.0001, max_iter=500, random_state=None, early_stopping=True, activation='relu', solver='adam'):
        self.hidden_layer_sizes = hidden_layer_sizes; self.alpha = alpha; self.max_iter = max_iter; self.random_state = random_state
        self.early_stopping = early_stopping; self.activation = activation; self.solver = solver
        self.estimator = None
    def fit(self, X, y):
        if isinstance(self.hidden_layer_sizes, str):
            try: self.hidden_layer_sizes = ast.literal_eval(self.hidden_layer_sizes)
            except: self.hidden_layer_sizes = (100,)
        self.estimator = MLPClassifier(hidden_layer_sizes=self.hidden_layer_sizes, alpha=self.alpha, max_iter=self.max_iter, 
                                       random_state=self.random_state, early_stopping=self.early_stopping, activation=self.activation, solver=self.solver)
        self.estimator.fit(X, y)
        self.classes_ = self.estimator.classes_
        return self
    def predict(self, X): return self.estimator.predict(X)
    def predict_proba(self, X): return self.estimator.predict_proba(X)

def parse_params(param_str):
    try: return ast.literal_eval(param_str)
    except: return {}

def find_optimal_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    return thresholds[np.argmax(tpr - fpr)]

def save_word_table(df, filename, title):
    doc = Document()
    doc.add_heading(title, level=1)
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = 'Table Grid'
    hdr_cells = table.rows[0].cells
    for i, col in enumerate(df.columns): 
        hdr_cells[i].text = str(col)
        for p in hdr_cells[i].paragraphs: 
            for r in p.runs: r.font.bold = True
    for _, row in df.iterrows():
        row_cells = table.add_row().cells
        for i, val in enumerate(row):
            row_cells[i].text = str(val)
    doc.save(os.path.join(OUTPUT_DIR, filename))
    print(f"📄 Table saved: {filename}")

# --- Main Logic ---
def main():
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

    print("=== 1. Loading Training Data (Internal Only) ===")
    try: df_train = pd.read_csv(TRAIN_FILE, encoding='gbk')
    except: df_train = pd.read_csv(TRAIN_FILE, encoding='utf-8')
    
    if 'Unnamed: 0' in df_train.columns: df_train.drop(columns=['Unnamed: 0'], inplace=True)
    if 'ID' in df_train.columns: df_train.drop(columns=['ID'], inplace=True) 
    
    y = df_train[TARGET_COL].astype(int)
    exclude = [TARGET_COL, 'Patient_ID', 'Center', 'AKIStage']
    X = df_train.drop(columns=[c for c in exclude if c in df_train.columns])
    X = pd.get_dummies(X, drop_first=True)
    
    print(f"  Training Set: {X.shape}")

    print("=== 2. Setting up Models & 10-fold CV ===")
    try: params_df = pd.read_excel(PARAMS_FILE)
    except: print("Error: Params file not found."); return

    models_map = {
        'LR': LogisticRegression(solver='liblinear', random_state=SEED),
        'DT': DecisionTreeClassifier(random_state=SEED),
        'RF': RandomForestClassifier(random_state=SEED),
        'KNN': KNeighborsClassifier(),
        'SVM': SVC(probability=True, random_state=SEED),
        'NB': GaussianNB(),
        'XGB': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED),
        'SGBT': GradientBoostingClassifier(random_state=SEED),
        'NNET': NNETWrapper(random_state=SEED)
    }

    # Prepare Pipelines with Best Params
    final_pipelines = {}
    
    for name, model in models_map.items():
        # Features
        feat_file = os.path.join(RFE_LISTS_FOLDER, f'selected_features_{name}_Combined.txt')
        valid_feats = X.columns.tolist()
        if os.path.exists(feat_file):
            with open(feat_file, 'r') as f: feats = [l.strip() for l in f if l.strip()]
            temp = [f for f in feats if f in X.columns]
            if temp: valid_feats = temp

        # Params
        p_row = params_df[params_df['Model'] == name]
        if not p_row.empty:
            p_dict = parse_params(p_row['Best Hyperparameters'].iloc[0])
            clean_p = {k.replace('classifier__', ''): v for k, v in p_dict.items()}
            if name == 'NNET' and 'hidden_layer_sizes_str' in clean_p:
                try: clean_p['hidden_layer_sizes'] = ast.literal_eval(clean_p.pop('hidden_layer_sizes_str'))
                except: clean_p['hidden_layer_sizes'] = (100,)
            for k in ['n_estimators', 'max_depth', 'min_samples_split', 'n_neighbors']:
                if k in clean_p and clean_p[k]: clean_p[k] = int(clean_p[k])
            try: model.set_params(**clean_p)
            except: pass

        pipeline = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])
        final_pipelines[name] = (pipeline, valid_feats)

    # --- 3. Rigorous 10-fold CV Loop ---
    print("\n=== 3. Running 10-fold Cross-Validation (This may take a moment) ===")
    
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    cv_metrics = {'Model': [], 'AUC': [], 'Spec': [], 'Sens': []}
    
    fold = 1
    for train_idx, val_idx in skf.split(X, y):
        print(f"  Processing Fold {fold}/10...", end='\r')
        
        # Data for this fold
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # 1. Train Base Models
        fold_models = {}
        fold_preds_val = {}
        
        # We also need OOF preds on the TRAIN part of this fold to train the Stacking Meta-Learner
        # To avoid another nested CV layer (which would be 10x10=100 runs), we will use 
        # cross_val_predict on the training portion of the fold just for the meta-learner.
        
        fold_oof_train_preds = pd.DataFrame(index=y_tr.index)
        
        for name, (pipe, feats) in final_pipelines.items():
            X_tr, X_val = X.iloc[train_idx][feats], X.iloc[val_idx][feats]
            
            # Clone and fit
            clf = clone(pipe)
            clf.fit(X_tr, y_tr)
            fold_models[name] = clf
            
            # Predict Validation Fold
            prob_val = clf.predict_proba(X_val)[:, 1]
            fold_preds_val[name] = prob_val
            
            # Record Metrics for Base Models
            auc_v = roc_auc_score(y_val, prob_val)
            th_v = find_optimal_threshold(y_val, prob_val)
            y_p_v = (prob_val >= th_v).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_val, y_p_v).ravel()
            
            cv_metrics['Model'].append(name)
            cv_metrics['AUC'].append(auc_v)
            cv_metrics['Spec'].append(tn/(tn+fp))
            cv_metrics['Sens'].append(tp/(tp+fn))
            
            # Generate OOF on Training Fold (for Stacking Meta training)
            # Using 3-fold internal CV for speed
            clf_oof = clone(pipe)
            oof_tr = cross_val_predict(clf_oof, X_tr, y_tr, cv=3, method='predict_proba', n_jobs=-1)[:, 1]
            fold_oof_train_preds[name] = oof_tr
        
        # 2. Ensemble - Voting
        # Identify Top 3 based on current fold performance (or global, but current is strictly safer)
        # Let's use current fold AUCs to be dynamic
        fold_aucs = {n: roc_auc_score(y_val, fold_preds_val[n]) for n in models_map}
        top3 = sorted(fold_aucs, key=fold_aucs.get, reverse=True)[:3]
        top4 = sorted(fold_aucs, key=fold_aucs.get, reverse=True)[:4] # For Stacking
        
        vote_prob = np.mean([fold_preds_val[n] for n in top3], axis=0)
        
        th_vote = find_optimal_threshold(y_val, vote_prob)
        y_p_vote = (vote_prob >= th_vote).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, y_p_vote).ravel()
        
        cv_metrics['Model'].append('Voting')
        cv_metrics['AUC'].append(roc_auc_score(y_val, vote_prob))
        cv_metrics['Spec'].append(tn/(tn+fp))
        cv_metrics['Sens'].append(tp/(tp+fn))
        
        # 3. Ensemble - Stacking (Optimized)
        # Train Meta-Learner on the OOF predictions of the Training Fold
        meta_X_tr = fold_oof_train_preds[top4]
        meta_X_val = pd.DataFrame({n: fold_preds_val[n] for n in top4})
        
        # Use LogisticRegressionCV for auto-tuning C inside the fold
        meta_learner = LogisticRegressionCV(cv=3, random_state=SEED, scoring='roc_auc')
        meta_learner.fit(meta_X_tr, y_tr)
        
        stack_prob = meta_learner.predict_proba(meta_X_val)[:, 1]
        
        th_stack = find_optimal_threshold(y_val, stack_prob)
        y_p_stack = (stack_prob >= th_stack).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, y_p_stack).ravel()
        
        cv_metrics['Model'].append('Stacking')
        cv_metrics['AUC'].append(roc_auc_score(y_val, stack_prob))
        cv_metrics['Spec'].append(tn/(tn+fp))
        cv_metrics['Sens'].append(tp/(tp+fn))
        
        fold += 1

    print("\n  Processing complete.")

    # === 4. Figure 3 (JAMA Style) ===
    print("\n=== Generating Figure 3 ===")
    df_plot = pd.DataFrame(cv_metrics)
    
    # Sort by AUC
    order = df_plot.groupby('Model')['AUC'].mean().sort_values(ascending=False).index
    df_plot['Model'] = pd.Categorical(df_plot['Model'], categories=order, ordered=True)
    df_plot = df_plot.sort_values('Model')
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 10), sharey=False)
    
    def draw_panel(ax, metric_col, title):
        sns.boxplot(data=df_plot, y='Model', x=metric_col, ax=ax, palette=COLORS, 
                    orient='h', showfliers=False, width=0.5, linewidth=1)
        sns.stripplot(data=df_plot, y='Model', x=metric_col, ax=ax, color='black', 
                      alpha=0.3, size=3, jitter=True)
        ax.set_title(title, fontsize=16, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_xlim(0, 1.1)
        ax.grid(axis='x', linestyle='--', alpha=0.5)
        
        stats = df_plot.groupby('Model')[metric_col].agg(['mean', 'std', 'sem'])
        stats['ci95_lo'] = stats['mean'] - 1.96 * stats['sem']
        stats['ci95_hi'] = stats['mean'] + 1.96 * stats['sem']
        
        for i, model in enumerate(order):
            row = stats.loc[model]
            text_mean = f"{row['mean']:.3f}±{row['std']:.3f}"
            text_ci = f"95% CI({row['ci95_lo']:.3f}-{row['ci95_hi']:.3f})"
            ax.text(0.05, i - 0.15, text_mean, fontsize=10, fontweight='bold', va='center')
            ax.text(0.05, i + 0.15, text_ci, fontsize=9, color='gray', va='center')

    draw_panel(axes[0], 'AUC', 'ROC')
    draw_panel(axes[1], 'Spec', 'Specificity')
    draw_panel(axes[2], 'Sens', 'Sensitivity')
    
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, 'Figure3_Internal_Validation.png'), dpi=300)
    plt.close()
    
    # === 5. Table Output ===
    summary = df_plot.groupby('Model').agg(['mean', 'std']).reset_index()
    summary.columns = ['Model'] + [f"{c[0]}_{c[1]}" for c in summary.columns[1:]]
    
    final_table = pd.DataFrame()
    final_table['Model'] = summary['Model']
    final_table['AUC'] = summary.apply(lambda r: f"{r['AUC_mean']:.3f} ± {r['AUC_std']:.3f}", axis=1)
    final_table['Specificity'] = summary.apply(lambda r: f"{r['Spec_mean']:.3f} ± {r['Spec_std']:.3f}", axis=1)
    final_table['Sensitivity'] = summary.apply(lambda r: f"{r['Sens_mean']:.3f} ± {r['Sens_std']:.3f}", axis=1)
    
    save_word_table(final_table, 'Table_Internal_Validation.docx', 'Internal Validation Results (10-fold CV)')

    # === Final Verdict ===
    best_row = summary.sort_values(by='AUC_mean', ascending=False).iloc[0]
    print("\n" + "="*60)
    print(f"🏆 INTERNAL Validation Best Model: 【 {best_row['Model']} 】")
    print(f"   AUC: {best_row['AUC_mean']:.4f} ± {best_row['AUC_std']:.4f}")
    print("="*60)
    print(f"Results saved in: {OUTPUT_DIR}")

if __name__ == '__main__':
    main()

ModuleNotFoundError: No module named 'docx'

In [ ]:
# -*- coding: utf-8 -*-
"""
Ultimate Edition: Internal Validation (10-fold CV) & Production Model Export
Features: 
1. Auto-loads hyperparameter CSV.
2. Generates JAMA-style Figure 3 and Word Table.
3. Trains and exports dual production models (Stacking & XGBoost) for Web App.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast
import warnings
import joblib  # 用于保存实战模型
from docx import Document # 请确保已通过 pip install python-docx 安装

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.base import BaseEstimator, ClassifierMixin, clone

# --- Plotting Style ---
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 300
warnings.filterwarnings('ignore')

# --- Configuration ---
BASE_DIR = os.getcwd()
TRAIN_FILE = os.path.join(BASE_DIR, 'imputation', 'imputed_data', 'train_imputed_random_forest.csv')
# 自动匹配你上传的 CSV 参数文件 (如果在同一级目录下)
PARAMS_FILE = os.path.join(BASE_DIR, 'All_Methods_Comparison_Summary.xlsx - Sheet1.csv')
RFE_LISTS_FOLDER = os.path.join(BASE_DIR, 'RFE_Final_Run', 'feature_lists')
OUTPUT_DIR = os.path.join(BASE_DIR, 'Final_Internal_Validation_Optimized')

TARGET_COL = 'PostopAKI'
SEED = 42

COLORS = {
    'LR': '#D55E00', 'DT': '#0072B2', 'RF': '#009E73', 
    'SVM': '#CC79A7', 'KNN': '#F0E442', 'NB': '#56B4E9', 
    'XGB': '#E69F00', 'SGBT': '#999999', 'NNET': '#000000',
    'Voting': '#882255', 'Stacking': '#117733'
}

# --- Helper Classes ---
class NNETWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_layer_sizes=(100,), alpha=0.0001, max_iter=500, random_state=None, early_stopping=True, activation='relu', solver='adam'):
        self.hidden_layer_sizes = hidden_layer_sizes; self.alpha = alpha; self.max_iter = max_iter; self.random_state = random_state
        self.early_stopping = early_stopping; self.activation = activation; self.solver = solver
        self.estimator = None
    def fit(self, X, y):
        if isinstance(self.hidden_layer_sizes, str):
            try: self.hidden_layer_sizes = ast.literal_eval(self.hidden_layer_sizes)
            except: self.hidden_layer_sizes = (100,)
        self.estimator = MLPClassifier(hidden_layer_sizes=self.hidden_layer_sizes, alpha=self.alpha, max_iter=self.max_iter, 
                                       random_state=self.random_state, early_stopping=self.early_stopping, activation=self.activation, solver=self.solver)
        self.estimator.fit(X, y)
        self.classes_ = self.estimator.classes_
        return self
    def predict(self, X): return self.estimator.predict(X)
    def predict_proba(self, X): return self.estimator.predict_proba(X)

def parse_params(param_str):
    try: return ast.literal_eval(param_str)
    except: return {}

def find_optimal_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    return thresholds[np.argmax(tpr - fpr)]

def save_word_table(df, filename, title):
    doc = Document()
    doc.add_heading(title, level=1)
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = 'Table Grid'
    hdr_cells = table.rows[0].cells
    for i, col in enumerate(df.columns): 
        hdr_cells[i].text = str(col)
        for p in hdr_cells[i].paragraphs: 
            for r in p.runs: r.font.bold = True
    for _, row in df.iterrows():
        row_cells = table.add_row().cells
        for i, val in enumerate(row):
            row_cells[i].text = str(val)
    doc.save(os.path.join(OUTPUT_DIR, filename))
    print(f"📄 Table saved: {filename}")

# --- Main Logic ---
def main():
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

    print("=== 1. Loading Training Data (Internal Only) ===")
    try: df_train = pd.read_csv(TRAIN_FILE, encoding='gbk')
    except: 
        try: df_train = pd.read_csv(TRAIN_FILE, encoding='utf-8')
        except Exception as e:
            print(f"❌ 找不到训练数据 {TRAIN_FILE}，请检查路径。")
            return
    
    if 'Unnamed: 0' in df_train.columns: df_train.drop(columns=['Unnamed: 0'], inplace=True)
    if 'ID' in df_train.columns: df_train.drop(columns=['ID'], inplace=True) 
    
    y = df_train[TARGET_COL].astype(int)
    exclude = [TARGET_COL, 'Patient_ID', 'Center', 'AKIStage']
    X = df_train.drop(columns=[c for c in exclude if c in df_train.columns])
    X = pd.get_dummies(X, drop_first=True)
    
    print(f"  Training Set: {X.shape}")

    print("\n=== 2. Setting up Models & Parameters ===")
    # 🌟 智能参数读取：优先读用户上传的 CSV
    try: 
        params_df = pd.read_csv(PARAMS_FILE)
        print("✅ 成功加载超参数配置文件！")
    except Exception as e: 
        print(f"⚠️ 找不到超参数文件 ({PARAMS_FILE})。自动使用各算法的【默认参数】继续...")
        params_df = pd.DataFrame() 

    models_map = {
        'LR': LogisticRegression(solver='liblinear', random_state=SEED),
        'DT': DecisionTreeClassifier(random_state=SEED),
        'RF': RandomForestClassifier(random_state=SEED),
        'KNN': KNeighborsClassifier(),
        'SVM': SVC(probability=True, random_state=SEED),
        'NB': GaussianNB(),
        'XGB': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED),
        'SGBT': GradientBoostingClassifier(random_state=SEED),
        'NNET': NNETWrapper(random_state=SEED)
    }

    final_pipelines = {}
    
    for name, model in models_map.items():
        # Feature selection list matching
        feat_file = os.path.join(RFE_LISTS_FOLDER, f'selected_features_{name}_Combined.txt')
        valid_feats = X.columns.tolist()
        if os.path.exists(feat_file):
            with open(feat_file, 'r') as f: feats = [l.strip() for l in f if l.strip()]
            temp = [f for f in feats if f in X.columns]
            if temp: valid_feats = temp

        # Parameter assigning with fallback safety
        if not params_df.empty and 'Model' in params_df.columns and 'Best Hyperparameters' in params_df.columns:
            p_row = params_df[params_df['Model'] == name]
            if not p_row.empty:
                p_dict = parse_params(p_row['Best Hyperparameters'].iloc[0])
                clean_p = {k.replace('classifier__', ''): v for k, v in p_dict.items()}
                if name == 'NNET' and 'hidden_layer_sizes_str' in clean_p:
                    try: clean_p['hidden_layer_sizes'] = ast.literal_eval(clean_p.pop('hidden_layer_sizes_str'))
                    except: clean_p['hidden_layer_sizes'] = (100,)
                for k in ['n_estimators', 'max_depth', 'min_samples_split', 'n_neighbors']:
                    if k in clean_p and clean_p[k]: clean_p[k] = int(clean_p[k])
                try: model.set_params(**clean_p)
                except: pass

        pipeline = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])
        final_pipelines[name] = (pipeline, valid_feats)

    # --- 3. Rigorous 10-fold CV Loop ---
    print("\n=== 3. Running 10-fold Cross-Validation ===")
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    cv_metrics = {'Model': [], 'AUC': [], 'Spec': [], 'Sens': []}
    
    fold = 1
    for train_idx, val_idx in skf.split(X, y):
        print(f"  Processing Fold {fold}/10...", end='\r')
        
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        fold_models = {}
        fold_preds_val = {}
        fold_oof_train_preds = pd.DataFrame(index=y_tr.index)
        
        for name, (pipe, feats) in final_pipelines.items():
            X_tr, X_val = X.iloc[train_idx][feats], X.iloc[val_idx][feats]
            clf = clone(pipe)
            clf.fit(X_tr, y_tr)
            fold_models[name] = clf
            
            prob_val = clf.predict_proba(X_val)[:, 1]
            fold_preds_val[name] = prob_val
            
            auc_v = roc_auc_score(y_val, prob_val)
            th_v = find_optimal_threshold(y_val, prob_val)
            y_p_v = (prob_val >= th_v).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_val, y_p_v).ravel()
            
            cv_metrics['Model'].append(name)
            cv_metrics['AUC'].append(auc_v)
            cv_metrics['Spec'].append(tn/(tn+fp))
            cv_metrics['Sens'].append(tp/(tp+fn))
            
            clf_oof = clone(pipe)
            oof_tr = cross_val_predict(clf_oof, X_tr, y_tr, cv=3, method='predict_proba', n_jobs=-1)[:, 1]
            fold_oof_train_preds[name] = oof_tr
        
        fold_aucs = {n: roc_auc_score(y_val, fold_preds_val[n]) for n in models_map}
        top3 = sorted(fold_aucs, key=fold_aucs.get, reverse=True)[:3]
        top4 = sorted(fold_aucs, key=fold_aucs.get, reverse=True)[:4]
        
        vote_prob = np.mean([fold_preds_val[n] for n in top3], axis=0)
        th_vote = find_optimal_threshold(y_val, vote_prob)
        y_p_vote = (vote_prob >= th_vote).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, y_p_vote).ravel()
        
        cv_metrics['Model'].append('Voting')
        cv_metrics['AUC'].append(roc_auc_score(y_val, vote_prob))
        cv_metrics['Spec'].append(tn/(tn+fp))
        cv_metrics['Sens'].append(tp/(tp+fn))
        
        meta_X_tr = fold_oof_train_preds[top4]
        meta_X_val = pd.DataFrame({n: fold_preds_val[n] for n in top4})
        
        meta_learner = LogisticRegressionCV(cv=3, random_state=SEED, scoring='roc_auc')
        meta_learner.fit(meta_X_tr, y_tr)
        
        stack_prob = meta_learner.predict_proba(meta_X_val)[:, 1]
        th_stack = find_optimal_threshold(y_val, stack_prob)
        y_p_stack = (stack_prob >= th_stack).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, y_p_stack).ravel()
        
        cv_metrics['Model'].append('Stacking')
        cv_metrics['AUC'].append(roc_auc_score(y_val, stack_prob))
        cv_metrics['Spec'].append(tn/(tn+fp))
        cv_metrics['Sens'].append(tp/(tp+fn))
        
        fold += 1

    print("\n  Processing complete.")

    # === 4. Figure 3 (JAMA Style) ===
    print("\n=== 4. Generating Figure 3 ===")
    df_plot = pd.DataFrame(cv_metrics)
    order = df_plot.groupby('Model')['AUC'].mean().sort_values(ascending=False).index
    df_plot['Model'] = pd.Categorical(df_plot['Model'], categories=order, ordered=True)
    df_plot = df_plot.sort_values('Model')
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 10), sharey=False)
    
    def draw_panel(ax, metric_col, title):
        sns.boxplot(data=df_plot, y='Model', x=metric_col, ax=ax, palette=COLORS, 
                    orient='h', showfliers=False, width=0.5, linewidth=1)
        sns.stripplot(data=df_plot, y='Model', x=metric_col, ax=ax, color='black', 
                      alpha=0.3, size=3, jitter=True)
        ax.set_title(title, fontsize=16, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_xlim(0, 1.1)
        ax.grid(axis='x', linestyle='--', alpha=0.5)
        
        stats = df_plot.groupby('Model')[metric_col].agg(['mean', 'std', 'sem'])
        stats['ci95_lo'] = stats['mean'] - 1.96 * stats['sem']
        stats['ci95_hi'] = stats['mean'] + 1.96 * stats['sem']
        
        for i, model in enumerate(order):
            row = stats.loc[model]
            text_mean = f"{row['mean']:.3f}±{row['std']:.3f}"
            text_ci = f"95% CI({row['ci95_lo']:.3f}-{row['ci95_hi']:.3f})"
            ax.text(0.05, i - 0.15, text_mean, fontsize=10, fontweight='bold', va='center')
            ax.text(0.05, i + 0.15, text_ci, fontsize=9, color='gray', va='center')

    draw_panel(axes[0], 'AUC', 'ROC')
    draw_panel(axes[1], 'Spec', 'Specificity')
    draw_panel(axes[2], 'Sens', 'Sensitivity')
    
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, 'Figure3_Internal_Validation.png'), dpi=300)
    plt.close()
    
    # === 5. Table Output ===
    summary = df_plot.groupby('Model').agg(['mean', 'std']).reset_index()
    summary.columns = ['Model'] + [f"{c[0]}_{c[1]}" for c in summary.columns[1:]]
    
    final_table = pd.DataFrame()
    final_table['Model'] = summary['Model']
    final_table['AUC'] = summary.apply(lambda r: f"{r['AUC_mean']:.3f} ± {r['AUC_std']:.3f}", axis=1)
    final_table['Specificity'] = summary.apply(lambda r: f"{r['Spec_mean']:.3f} ± {r['Spec_std']:.3f}", axis=1)
    final_table['Sensitivity'] = summary.apply(lambda r: f"{r['Sens_mean']:.3f} ± {r['Sens_std']:.3f}", axis=1)
    
    save_word_table(final_table, 'Table_Internal_Validation.docx', 'Internal Validation Results (10-fold CV)')

    best_row = summary.sort_values(by='AUC_mean', ascending=False).iloc[0]
    print("\n" + "="*60)
    print(f"🏆 INTERNAL Validation Best Model: 【 {best_row['Model']} 】")
    print(f"   AUC: {best_row['AUC_mean']:.4f} ± {best_row['AUC_std']:.4f}")
    print("="*60)
    print(f"Results saved in: {OUTPUT_DIR}")

    # =====================================================================
    # === 6. Training & Saving Final Production Models (For Web App) ===
    # =====================================================================
    print("\n=== 6. Training & Saving Final Production Models ===")
    
    # 【提取用于前端网页的 12 个临床核心特征】
    web_features = ['Age', 'Gender', 'BMI', 'Diabetes', 'PreopAlb', 'OperationTime', 
                    'NonOpTransfusion', 'IntraopPlasma', 'NonOpAlbumin', 'PerioperativeVasoactive', 
                    'TNM_Stage', 'Gastrocolorectal']
    
    # 安全匹配真实列名
    available_cols = []
    for f in web_features:
        matched = [c for c in df_train.columns if f.lower() in c.lower()]
        if matched: available_cols.append(matched[0])
    
    if len(available_cols) > 0:
        X_web = df_train[available_cols].fillna(0)
        
        print("Training final Stacking model for Overall AKI on 100% data...")
        # 定义底层模型 (用 RF, XGB, LR 作为强大的基石)
        base_estimators = [
            ('rf', RandomForestClassifier(n_estimators=150, max_depth=10, random_state=SEED, class_weight='balanced')),
            ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss', max_depth=6, random_state=SEED)),
            ('lr', LogisticRegression(solver='liblinear', class_weight='balanced', random_state=SEED))
        ]
        # 定义顶层元学习器
        final_stacking_clf = StackingClassifier(estimators=base_estimators, final_estimator=LogisticRegression(), cv=5)
        final_stacking_clf.fit(X_web, y)
        
        stacking_path = os.path.join(OUTPUT_DIR, 'model_stacking.pkl')
        # 连同特征列名一起打包保存，保证网页端读取绝对安全
        joblib.dump((final_stacking_clf, available_cols), stacking_path) 
        print(f"✅ Stacking 部署模型已保存至: {stacking_path}")

        if 'AKIStage' in df_train.columns:
            print("Training final XGBoost model for Severe AKI...")
            y_severe = (df_train['AKIStage'] >= 2).astype(int)
            final_xgb_severe = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, use_label_encoder=False, eval_metric='logloss', random_state=SEED)
            final_xgb_severe.fit(X_web, y_severe)
            
            xgb_path = os.path.join(OUTPUT_DIR, 'model_xgboost_severe.pkl')
            joblib.dump((final_xgb_severe, available_cols), xgb_path)
            print(f"✅ XGBoost 重症模型已保存至: {xgb_path}")
        else:
            print("⚠️ 找不到 'AKIStage' 列，跳过重症模型训练。")
    else:
        print("⚠️ 无法在数据集中找到网页所需的12个特征，部署模型生成失败。")

    print("\n🎉 全部运行完成！你可以去文件夹里拿 .pkl 文件跑网页应用了！")

if __name__ == '__main__':
    main()

=== 1. Loading Training Data (Internal Only) ===
  Training Set: (2446, 138)

=== 2. Setting up Models & Parameters ===
⚠️ 找不到超参数文件 (/home/lei/AKI新分析/All_Methods_Comparison_Summary.xlsx - Sheet1.csv)。自动使用各算法的【默认参数】继续...

=== 3. Running 10-fold Cross-Validation ===
  Processing Fold 10/10...
  Processing complete.

=== 4. Generating Figure 3 ===
📄 Table saved: Table_Internal_Validation.docx

🏆 INTERNAL Validation Best Model: 【 Voting 】
   AUC: 0.8324 ± 0.0544
Results saved in: /home/lei/AKI新分析/Final_Internal_Validation_Optimized

=== 6. Training & Saving Final Production Models ===
Training final Stacking model for Overall AKI on 100% data...
